# KRAS Allele-Specific Gene Dependencies in COAD — One-Way ANOVA

This notebook compares CRISPR gene effect scores across three common KRAS alleles in colorectal adenocarcinoma (COAD) — **G12D**, **G12V**, and **A146T** — against a KRAS wild-type baseline, using DepMap 24Q2 data.

## Motivation

The companion notebook (`DepMap_GSEA.ipynb`) identifies genes and pathways that are differentially essential in KRAS G12D COAD lines versus WT. However, it does not address whether those dependencies are specific to G12D or are shared across all KRAS mutant alleles.

KRAS alleles differ in their intrinsic GTP hydrolysis rates and effector binding preferences:
- **G12D**: impaired GAP-mediated GTP hydrolysis; preferential PI3K activation
- **G12V**: impaired hydrolysis; strongest intrinsic activity; associated with poor prognosis
- **A146T**: faster nucleotide exchange; weaker transformation potential; distinct signaling profile

If these biochemical differences produce distinct gene dependencies, that would suggest allele-specific therapeutic strategies are warranted.

## Approach

For each gene measured in the DepMap CRISPR screens, we run a **one-way ANOVA** across the four groups (G12D, G12V, A146T, WT) to identify genes where the mean effect score differs significantly across genotypes. Genes with significant F-statistics are candidates for allele-specific dependencies.

**Data:** DepMap Public 24Q2  
**Data DOI:** [10.25452/figshare.plus.27993248.v1](https://doi.org/10.25452/figshare.plus.27993248.v1)

## Setup

Update the paths in the config cell to point to your local DepMap data files.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.stats
import numpy as np
import gseapy as gp
from statsmodels.stats.multitest import multipletests
from pathlib import Path

In [ ]:
# --- Configure paths ---
DATA_DIR = Path("/path/to/depmap_24Q2")

MODEL_PATH      = DATA_DIR / "Model.csv"
OSM_PATH        = DATA_DIR / "OmicsSomaticMutations.csv"
EFFECT_PATH     = DATA_DIR / "CRISPRGeneEffect.csv"
DEPENDENCY_PATH = DATA_DIR / "CRISPRGeneDependency.csv"

# --- Analysis parameters ---
CANCER_TYPE     = "COAD"
GENE            = "KRAS"
KRAS_ALLELES    = ['p.G12D', 'p.G12V', 'p.A146T']

# Output directory
REPO_ROOT  = Path().resolve().parent
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

ANOVA_PVAL_CUTOFF = 0.01  # raw p-value threshold before FDR correction

In [ ]:
# Load DepMap data
model_df      = pd.read_csv(MODEL_PATH,      header=0, index_col=0)
osm_df        = pd.read_csv(OSM_PATH,        header=0, low_memory=False)
effect_df = pd.read_csv(EFFECT_PATH, header=0)
effect_df.rename(columns={effect_df.columns[0]: 'ModelID'}, inplace=True)
effect_df.columns = ['ModelID'] + [col.split(' ')[0] for col in effect_df.columns[1:]]
dependency_df = pd.read_csv(DEPENDENCY_PATH, header=0, index_col=0)

print(f"Loaded {effect_df.shape[0]} cell lines x {effect_df.shape[1]} genes")

## 1. Define Genotype Groups

We create four groups of COAD cell lines:
- **G12D**, **G12V**, **A146T**: lines carrying each specific KRAS allele
- **WT**: COAD lines with *no* KRAS mutation of any kind

Cell lines with KRAS alleles other than the three studied are excluded entirely, to keep the WT group clean.

In [ ]:
# All COAD models
coad_models = model_df[model_df['OncotreeCode'] == CANCER_TYPE].copy()
coad_models = coad_models.reset_index()[['ModelID']]

# Merge with somatic mutation data
osm_coad = coad_models.merge(osm_df, on='ModelID', how='inner')
kras_osm = osm_coad[osm_coad['HugoSymbol'] == GENE]

# Build one group per allele
mutation_groups = {}
for allele in KRAS_ALLELES:
    allele_ids = (
        kras_osm[kras_osm['ProteinChange'] == allele][['ModelID']]
        .drop_duplicates()
    )
    effect_allele = allele_ids.merge(effect_df, on='ModelID', how='inner').set_index('ModelID')
    mutation_groups[allele] = effect_allele
    print(f"{allele}: {len(effect_allele)} lines")

# WT group: COAD lines with no KRAS mutation at all
all_kras_ids = kras_osm['ModelID'].unique()
wt_ids = coad_models[~coad_models['ModelID'].isin(all_kras_ids)]
effect_wt = wt_ids.merge(effect_df, on='ModelID', how='inner').set_index('ModelID')
mutation_groups['WT'] = effect_wt
print(f"WT: {len(effect_wt)} lines")

## 2. One-Way ANOVA Across Allele Groups

For each gene, we test whether any of the four group means differ using `scipy.stats.f_oneway`. A significant F-statistic (after FDR correction) indicates that the gene's essentiality varies across KRAS genotypes.

**Caveat:** ANOVA tests for *any* difference across groups, but does not specify *which* groups differ. Post-hoc pairwise tests (e.g. Tukey HSD) would be needed to determine whether a hit is truly allele-specific or just driven by the mutant-vs-WT contrast. With the current sample sizes, post-hoc tests have limited power and are not shown here.

In [ ]:
anova_results = []

for gene in effect_df.columns:
    # Get effect scores for each group for this gene
    groups = [
        mutation_groups[g][gene].dropna()
        for g in mutation_groups
        if gene in mutation_groups[g].columns
    ]
    # Need at least 2 groups with >1 observation
    groups = [g for g in groups if len(g) > 1]
    if len(groups) < 2:
        continue

    f_stat, p_value = scipy.stats.f_oneway(*groups)
    anova_results.append({'gene': gene, 'f_stat': f_stat, 'p_value': p_value})

anova_df = pd.DataFrame(anova_results).dropna().set_index('gene')

# Benjamini-Hochberg FDR correction
reject, fdr_vals, _, _ = multipletests(anova_df['p_value'], alpha=0.25, method='fdr_bh')
anova_df['fdr']         = fdr_vals
anova_df['significant'] = reject

sig_anova = anova_df[anova_df['significant']].sort_values('p_value')
print(f"Genes significant at FDR < 0.25: {len(sig_anova)}")
sig_anova.head(15)

## 3. Visualizing Top Hits

For the top ANOVA hits, we plot the distribution of CRISPR effect scores across all four groups. Genes with a mean effect score below ~-0.5 are considered likely essential (the standard DepMap threshold is approximately -0.5 Chronos score). A gene that is more negative specifically in one mutant group suggests a genotype-specific dependency.

We highlight the top hits by mean effect in G12D (most essential first), which links back to the GSEA findings.

In [ ]:
# Add mean effect score per allele for each significant gene
for allele, label in zip(KRAS_ALLELES + ['WT'], ['mean_G12D', 'mean_G12V', 'mean_A146T', 'mean_WT']):
    grp = mutation_groups[allele if allele in mutation_groups else 'WT']
    key = allele if allele != 'WT' else 'WT'
    means = mutation_groups[key].mean(axis=0)
    sig_anova[label] = means

sig_anova = sig_anova.sort_values('mean_G12D', ascending=True)
print("Top 10 hits by mean G12D effect score:")
sig_anova[['f_stat', 'p_value', 'fdr', 'mean_G12D', 'mean_G12V', 'mean_A146T', 'mean_WT']].head(10)

In [ ]:
# Box plots for top 4 hits by mean G12D effect
top_genes = [g for g in sig_anova.head(5).index.tolist() if g != 'KRAS'][:4]

group_labels = KRAS_ALLELES + ['WT']
colors = ['#E07B54', '#5B8DB8', '#6AAB6E', '#AAAAAA']

fig, axes = plt.subplots(1, 4, figsize=(14, 5), sharey=False)

for ax, gene in zip(axes, top_genes):
    gene_col = gene  # columns already stripped of Entrez IDs
    data_by_group = []
    for grp_key in group_labels:
        grp_df = mutation_groups[grp_key]
        if gene_col in grp_df.columns:
            vals = grp_df[gene_col].dropna().values
        else:
            vals = np.array([])
        data_by_group.append(vals)

    bp = ax.boxplot(
        data_by_group,
        patch_artist=True,
        widths=0.5,
        medianprops=dict(color='black', linewidth=1.5),
        flierprops=dict(marker='o', markersize=4, alpha=0.5)
    )

    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    # Overlay individual data points
    for i, vals in enumerate(data_by_group):
        jitter = np.random.uniform(-0.12, 0.12, size=len(vals))
        ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                   color=colors[i], s=22, alpha=0.8, zorder=3, edgecolors='white', linewidths=0.4)

    ax.axhline(y=-0.5, color='red', linestyle='--', linewidth=0.8, alpha=0.6, label='Essential threshold')
    ax.set_xticks(range(1, len(group_labels) + 1))
    ax.set_xticklabels(['G12D', 'G12V', 'A146T', 'WT'], fontsize=9)
    ax.set_title(gene, fontsize=11, fontweight='bold')
    ax.set_ylabel('CRISPR Gene Effect (Chronos)' if ax == axes[0] else '', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Annotate with FDR
    fdr_val = sig_anova.loc[gene, 'fdr']
    ax.text(0.97, 0.02, f'FDR={fdr_val:.2e}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=7.5, color='#555555')

axes[0].legend(fontsize=8, loc='upper right')
fig.suptitle(f'CRISPR Gene Effect by KRAS Genotype — {CANCER_TYPE} (DepMap 24Q2)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "anova_boxplots.png", dpi=150, bbox_inches='tight')
plt.show()

## 4. Interpretation

The ANOVA identifies genes whose essentiality varies across KRAS genotypes in COAD. The top hits by F-statistic include core cellular machinery genes, notably ribosomal subunits, consistent with the GSEA findings in the companion notebook.

Several observations are worth noting:
- The signal is predominantly driven by the **G12D vs. WT** contrast, given G12D's higher sample size and stronger effect sizes
- **A146T** lines show weaker and less consistent dependencies, potentially reflecting its distinct signaling profile
- Ribosomal gene essentiality in G12D is consistent with reports of elevated ribosome biogenesis in RAS-driven cancers and the role of p53-ribosomal stress signaling in this context

Future directions would include post-hoc pairwise tests (Tukey HSD) and expanding to other cancer types (PAAD, LUAD) to assess tissue specificity of these allele-specific dependencies.